# 한국어 회의 메모 요약 서비스 — 실행/테스트 노트북

이 노트북은 프로젝트를 **처음부터 끝까지 실행하고 테스트**하기 위한 것입니다.
위에서부터 셀을 순서대로 실행하세요.

구성:
1. 환경 확인
2. 서버 실행 도우미 정의
3. (선택) 모델 단독 테스트
4. FastAPI 서버 실행
5. API 테스트 (health / 401 / 422 / 빠름 / 정확)
6. 서버 종료

> ⚠️ **코드를 수정했다면** 커널을 재시작(Kernel → Restart)한 뒤 맨 위부터 다시 실행하세요.
> 메모리에 남은 이전 코드 때문에 변경이 반영되지 않을 수 있습니다.

## 1. 환경 확인
현재 파이썬과 작업 폴더를 확인합니다. `app/` 폴더가 보여야 합니다.

In [ ]:
import sys, os

print("python:", sys.executable)   # .venv 의 python.exe 여야 함
print("작업 폴더:", os.getcwd())
print("app 폴더 존재:", os.path.isdir("app"))  # True 여야 함

## 2. 서버 실행 도우미 정의
노트북 안에서 uvicorn 서버를 띄우고 멈추는 함수입니다. **한 번만 실행**하면 됩니다.
- `serve_in_thread("app.main:app")` : 서버 시작
- `stop_server(8000)` : 서버 종료

In [ ]:
import os, sys, asyncio, threading, time, socket, contextlib
import uvicorn

# 코드 폴더가 없으면 만들어 둔다 (보통 이미 있음)
for _d in ("app", "models", "data", "frontend"):
    os.makedirs(_d, exist_ok=True)

_SERVERS = {}  # port -> (server, thread)

def _port_open(host, port):
    with contextlib.closing(socket.socket()) as s:
        s.settimeout(0.5)
        return s.connect_ex((host, port)) == 0

def stop_server(port=8000):
    """실행 중인 서버를 멈춥니다."""
    entry = _SERVERS.pop(port, None)
    if not entry:
        return
    server, thread = entry
    server.should_exit = True
    for _ in range(50):
        if not thread.is_alive():
            break
        time.sleep(0.1)

def serve_in_thread(app, host="127.0.0.1", port=8000, log_level="warning"):
    """백그라운드 스레드에서 uvicorn 서버를 띄웁니다.

    app : 'app.main:app' 같은 import 경로.
    같은 포트에 서버가 있으면 먼저 멈추고 새로 띄웁니다."""
    stop_server(port)
    if _port_open(host, port):
        print(f"⚠️ 포트 {port}를 다른 프로세스가 사용 중입니다. 해당 서버를 종료하고 다시 실행하세요.")
        return None
    if isinstance(app, str):
        sys.modules.pop(app.split(":")[0], None)   # 파일을 다시 저장한 경우 최신 내용 반영
    config = uvicorn.Config(app, host=host, port=port, log_level=log_level, loop="asyncio")
    server = uvicorn.Server(config)
    server.install_signal_handlers = lambda: None
    def _run():
        loop = asyncio.SelectorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)
        loop.run_until_complete(server.serve())
    thread = threading.Thread(target=_run, daemon=True)
    thread.start()
    _SERVERS[port] = (server, thread)
    # 모델 로드 때문에 기동이 느릴 수 있어 최대 5분 대기
    for i in range(600):
        if _port_open(host, port):
            print(f"✅ 서버 실행됨: http://{host}:{port}")
            return server
        if not thread.is_alive():
            print("❌ 서버 스레드가 종료됐습니다. 로그를 확인하세요.")
            return server
        if i > 0 and i % 20 == 0:
            print(f"  ... 모델 로드 중 ({i//2}초 경과)")
        time.sleep(0.5)
    print("⏱️ 5분 내에 서버가 시작되지 않았습니다.")
    return server

print("도우미 준비 완료 (serve_in_thread, stop_server)")

## 3. (선택) 모델 단독 테스트
서버 없이 빠름(t5) 모델만으로 요약이 되는지 먼저 확인합니다.
처음 실행 시 모델을 다운로드하므로 시간이 걸릴 수 있습니다.

In [ ]:
from transformers import pipeline

summarizer = pipeline("summarization", model="eenzeenee/t5-base-korean-summarization")

sample = (
    "오늘 회의에서는 프로젝트 일정 지연 원인과 대응 방안에 대해 논의하였다. "
    "주요 지연 원인은 부품 입고 일정 변경, 설계 검토 지연, 일부 제작 공정의 병목으로 확인되었다. "
    "이에 따라 관련 부서에서는 납기 영향도를 재검토하고, 고객사에는 변경 가능성이 있는 일정에 대해 사전 안내하기로 하였다."
)
# truncation=True : 입력이 길면 자동으로 잘라 길이 경고 방지
out = summarizer("summarize: " + sample, max_length=120, min_length=30,
                 do_sample=False, truncation=True, num_beams=6,
                 no_repeat_ngram_size=3, length_penalty=2.0)
print(out[0]["summary_text"])

## 4. FastAPI 서버 실행
`app/main.py` 의 서버를 띄웁니다. 시작 시 빠름(t5) 모델을 로드합니다.
(정확 모드 LLM 은 무거우므로 첫 요청 때 로드됩니다.)

In [ ]:
serve_in_thread("app.main:app", port=8000)

## 5. API 테스트
서버가 떴으면 아래 셀들로 동작을 확인합니다.
API Key 는 `my-secret-key` 입니다.

### 5-1. /health (인증 불필요)

In [ ]:
import requests

API = "http://127.0.0.1:8000"
KEY = {"X-API-Key": "my-secret-key"}

print(requests.get(f"{API}/health").json())

### 5-2. API Key 없이 호출 → 401 기대

In [ ]:
r = requests.post(f"{API}/predict", json={"text": sample})
print("상태:", r.status_code)   # 401
print(r.json())

### 5-3. 잘못된 API Key → 401 기대

In [ ]:
r = requests.post(f"{API}/predict", json={"text": sample}, headers={"X-API-Key": "wrong-key"})
print("상태:", r.status_code)   # 401
print(r.json())

### 5-4. 너무 짧은 입력(30자 미만) → 422 기대

In [ ]:
r = requests.post(f"{API}/predict", json={"text": "너무 짧은 글"}, headers=KEY)
print("상태:", r.status_code)   # 422
print(r.json())

### 5-5. 빠름(fast) 모드 → 200 기대
t5 모델, 수 초 소요.

In [ ]:
import time

t0 = time.time()
r = requests.post(f"{API}/predict", json={"text": sample, "mode": "fast"}, headers=KEY)
print("상태:", r.status_code, f"/ 소요 {time.time()-t0:.1f}초")
d = r.json()
print("요약:", d["summary"])
print(f"원문 {d['original_length']}자 → 요약 {d['summary_length']}자 / 모델 {d['model_name']}")

### 5-6. 정확(accurate) 모드 → 200 기대
지시형 LLM(Qwen). **CPU에서 1~3분** 걸립니다. 첫 요청은 모델 로드(최초 1회 ~6GB 다운로드) 때문에 더 깁니다.

In [ ]:
t0 = time.time()
r = requests.post(f"{API}/predict", json={"text": sample, "mode": "accurate"},
                  headers=KEY, timeout=600)
print("상태:", r.status_code, f"/ 소요 {time.time()-t0:.1f}초")
d = r.json()
print("요약:", d["summary"])
print(f"원문 {d['original_length']}자 → 요약 {d['summary_length']}자 / 모델 {d['model_name']}")

## 6. 서버 종료
테스트가 끝나면 서버를 멈춥니다.

In [ ]:
stop_server(8000)
print("서버 종료됨")